# 17 — Interrogate Non-Linearity

Verifies that LASSO feature selection (Notebook 15) did not permanently discard
non-linear signals, and characterises the non-linearity of the features XGBoost
(Notebook 16) actually uses.

## Sections

1. **Linearity trap audit table** — LASSO vs. MI rank divergence per feature/outcome
2. **PDP + ICE for top LASSO-selected features** — shape of marginal effects
3. **PDP for MI-rescued features** — validates whether non-linear pattern is real
4. **SHAP interaction values** — joint effects LASSO cannot detect (top-20 features)
5. **Monotonicity check** — SHAP sign vs. domain expectation for key features
6. **Direction concordance table** — LASSO coefficient sign vs. SHAP mean sign

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER   (default: 'data')
```

In [ ]:
import os
import json
import re
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.inspection import PartialDependenceDisplay

import xgboost as xgb
import shap

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")

TRAIN_END_YEAR = 2018
VAL_END_YEAR   = 2021

OUTCOMES = [
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
]

# Domain-theory expected sign for key features (positive = increases instability risk)
THEORY_SIGNS = {
    "gdp_pc":                    -1,  # richer → more stable
    "gdp_pc_lag1":               -1,
    "polity2":                   -1,  # more democratic → lower coup risk
    "polity2_lag1":              -1,
    "food_price_index":          +1,  # higher food prices → more unrest
    "food_price_index_lag1":     +1,
    "regime_durability":         -1,  # older regime → lower transition risk
    "durable":                   -1,
    "polity_anocracy_flag":      +1,  # mid-range polity → higher risk
    "arch_irregular_exit_count": +1,  # past irregular exits → higher coup risk
    "arch_irregular_exit_count_lag1": +1,
    "vdem_v2x_libdem":           -1,  # more liberal democracy → lower backsliding risk
    "vdem_v2x_libdem_lag1":      -1,
    "ucdp_battle_deaths":        +1,  # ongoing violence → more onset risk
    "cnts_domestic_conflict_index": +1,
}

OUTCOME_COLS = [
    "civil_war_onset", "coup_attempt", "regime_backsliding",
    "mass_unrest_onset", "humanitarian_crisis_onset",
]
ID_COLS = ["iso3", "year"]

print(f"Outcomes       : {OUTCOMES}")
print(f"Theory signs   : {len(THEORY_SIGNS)} features")

## ADLS helpers and data loading

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

_fs = adlfs.AzureBlobFileSystem(account_name=ADLS_ACCOUNT_NAME, credential=credential)

def _latest_date_dir(prefix: str) -> str | None:
    full = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = _fs.ls(full, detail=False)
    except FileNotFoundError:
        return None
    dirs = sorted([e for e in entries if re.search(r'/\d{8}(/|$)', e)], reverse=True)
    return dirs[0] if dirs else None

def read_parquet_from_adls(path: str) -> pd.DataFrame:
    full = (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + path.replace(f"{ADLS_CONTAINER}/", "", 1)
    )
    return pd.read_parquet(full, storage_options=storage_options)

def read_json_from_adls(path: str) -> dict | None:
    try:
        with _fs.open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return None

def load_xgb_model(outcome: str) -> xgb.XGBClassifier | None:
    model_dir = _latest_date_dir(f"models")
    if not model_dir:
        return None
    files = _fs.glob(f"{model_dir}/{outcome}/model.json")
    if not files:
        return None
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
        with _fs.open(files[0], "rb") as src:
            tmp.write(src.read())
        tmp_path = tmp.name
    model = xgb.XGBClassifier()
    model.load_model(tmp_path)
    return model

# ── Load feature matrix and labels ───────────────────────────────────────────
_fm_dir = _latest_date_dir("processed/feature_matrix")

def _read_named(filename: str) -> pd.DataFrame | None:
    if not _fm_dir:
        return None
    files = _fs.glob(f"{_fm_dir}/{filename}")
    return read_parquet_from_adls(files[0]) if files else None

df_features = _read_named("feature_matrix.parquet")
df_labels   = _read_named("labels.parquet")

if df_features is not None:
    print(f"Feature matrix : {df_features.shape}")
    print(f"Labels         : {df_labels.shape if df_labels is not None else 'not loaded'}")
else:
    print("ERROR: could not load feature matrix")